In [2]:
import requests

url = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population_density"
headers = {"User-Agent": "SIS-project"}
response = requests.get(url, headers=headers)
print(response.status_code)

200


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table", class_="wikitable")
print(len(tables))

1


In [4]:
table = tables[0]
rows = table.find_all("tr")
print(len(rows))
print(rows[0].text)
print(rows[1].text)

248
Location
Density
Population
Land area
Ref.

/km2
/mi2
km2
mi2


In [5]:
print(rows[2].text)
print(rows[3].text)


 World551408,231,613,070148,940,00057,510,000[b]

 Macau (China)21,82656,530720,2623313



In [6]:
cells = rows[3].find_all(["th", "td"])
print(len(cells))
for c in cells:
    print(repr(c.text))

7
'\xa0Macau (China)'
'21,826'
'56,530'
'720,262'
'33'
'13'
''


In [7]:
data = []
for row in rows[3:]:
    cells = row.find_all(["th", "td"])
    if len(cells) < 5:
        continue
    data.append([cells[0].text, cells[1].text, cells[3].text, cells[4].text])

print(len(data))
print(data[0])

245
['\xa0Macau (China)', '21,826', '720,262', '33']


In [8]:
import pandas as pd

df = pd.DataFrame(data, columns=["Country", "Density", "Population", "Area"])
print(df.head())

              Country Density Population   Area
0       Macau (China)  21,826    720,262     33
1              Monaco  19,428     38,857      2
2           Singapore   8,290  6,110,200    737
3   Hong Kong (China)   7,062  7,414,910  1,050
4      Gibraltar (UK)   5,800     39,329    6.8


In [9]:
df["Country"] = df["Country"].str.replace("\xa0", " ").str.strip()
print(df.head())

             Country Density Population   Area
0      Macau (China)  21,826    720,262     33
1             Monaco  19,428     38,857      2
2          Singapore   8,290  6,110,200    737
3  Hong Kong (China)   7,062  7,414,910  1,050
4     Gibraltar (UK)   5,800     39,329    6.8


In [10]:
for col in ["Density", "Population", "Area"]:
    df[col] = df[col].str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.head())
print(df.dtypes)

             Country  Density  Population    Area
0      Macau (China)  21826.0    720262.0    33.0
1             Monaco  19428.0     38857.0     2.0
2          Singapore   8290.0   6110200.0   737.0
3  Hong Kong (China)   7062.0   7414910.0  1050.0
4     Gibraltar (UK)   5800.0     39329.0     6.8
Country           str
Density       float64
Population    float64
Area          float64
dtype: object


In [11]:
print(df.isna().sum())
print(df.duplicated().sum())

Country       0
Density       0
Population    1
Area          0
dtype: int64
0


In [12]:
print(df[df["Population"].isna()])

        Country  Density  Population        Area
244  Antarctica    0.009         NaN  14200000.0


In [13]:
df = df.dropna(subset=["Population"])
print(len(df))

244


In [14]:
print(df[df["Country"].str.contains(r"\(")].shape[0])
print(df[df["Country"].str.contains(r"\(")]["Country"].head(15))

42
0                   Macau (China)
3               Hong Kong (China)
4                  Gibraltar (UK)
10              Sint Maarten (NL)
11                   Bermuda (UK)
13                    Jersey (UK)
14                  Guernsey (UK)
18                     Aruba (NL)
22      Saint Barthélemy (France)
27          Saint Martin (France)
32                   Curaçao (NL)
36               Puerto Rico (US)
40                      Guam (US)
41            Cayman Islands (UK)
53    British Virgin Islands (UK)
Name: Country, dtype: str


In [15]:
df["Country"] = df["Country"].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
print(df["Country"].head(15))

0            Macau
1           Monaco
2        Singapore
3        Hong Kong
4        Gibraltar
5     Vatican City
6          Bahrain
7         Maldives
8            Malta
9       Bangladesh
10    Sint Maarten
11         Bermuda
12       Palestine
13          Jersey
14        Guernsey
Name: Country, dtype: str


In [16]:
df.to_csv("wikipedia_countries.csv", index=False)
print(df.shape)

(244, 4)


In [17]:
wb = pd.read_csv("world_bank_economic_2023.xls")
print(wb.shape)
print(wb.head())

(217, 8)
  iso3         Country  year  gdp_per_capita  life_expectancy  unemployment  \
0  AFG     Afghanistan  2023      413.757895           66.035        14.008   
1  ALB         Albania  2023     9740.702341           79.602        10.669   
2  DZA         Algeria  2023     5370.477235           76.261        11.645   
3  ASM  American Samoa  2023             NaN           72.852           NaN   
4  AND         Andorra  2023    46812.448450           84.041           NaN   

   population   log_gdp  
0  41454761.0  2.616746  
1   2411658.0  3.988590  
2  46164219.0  3.730013  
3     47521.0       NaN  
4     80856.0  4.670361  


In [18]:
merged = pd.merge(df, wb, on="Country", how="outer", indicator=True)
print(merged["_merge"].value_counts())

_merge
both          180
left_only      64
right_only     37
Name: count, dtype: int64


In [19]:
print(merged[merged["_merge"] == "right_only"]["Country"].tolist())

['Bahamas, The', 'Brunei Darussalam', 'Cabo Verde', 'Channel Islands', 'Congo, Dem. Rep.', 'Congo, Rep.', "Cote d'Ivoire", 'Curacao', 'Czechia', 'Egypt, Arab Rep.', 'Gambia, The', 'Hong Kong SAR, China', 'Iran, Islamic Rep.', "Korea, Dem. People's Rep.", 'Korea, Rep.', 'Kyrgyz Republic', 'Lao PDR', 'Macao SAR, China', 'Micronesia, Fed. Sts.', 'Naoero', 'Puerto Rico (US)', 'Russian Federation', 'Sao Tome and Principe', 'Sint Maarten (Dutch part)', 'Slovak Republic', 'Somalia, Fed. Rep.', 'St. Kitts and Nevis', 'St. Lucia', 'St. Martin (French part)', 'St. Vincent and the Grenadines', 'Syrian Arab Republic', 'Turkiye', 'Venezuela, RB', 'Viet Nam', 'Virgin Islands (U.S.)', 'West Bank and Gaza', 'Yemen, Rep.']


In [20]:
print(merged[merged["_merge"] == "left_only"]["Country"].tolist())

['Akrotiri and Dhekelia', 'Anguilla', 'Bahamas', 'Bonaire', 'Brunei', 'Cape Verde', 'Christmas Island', 'Cocos Islands', 'Congo', 'Cook Islands', 'Curaçao', 'Czech Republic', 'DR Congo', 'Egypt', 'Falkland Islands', 'Federated States of Micronesia', 'Gambia', 'Guernsey', 'Hong Kong', 'Iran', 'Ivory Coast', 'Jersey', 'Kyrgyzstan', 'Laos', 'Macau', 'Montserrat', 'Nauru', 'Niue', 'Norfolk Island', 'North Korea', 'Northern Cyprus', 'Palestine', 'Pitcairn Islands', 'Puerto Rico', 'Russia', 'Saba', 'Saint Barthélemy', 'Saint Helena, Ascension and Tristan da Cunha', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Martin', 'Saint Pierre and Miquelon', 'Saint Vincent and the Grenadines', 'Sint Eustatius', 'Sint Maarten', 'Slovakia', 'Somalia', 'Somaliland', 'South Korea', 'South Ossetia', 'Syria', 'São Tomé and Príncipe', 'Taiwan', 'Tokelau', 'Transnistria', 'Turkey', 'U.S. Virgin Islands', 'Vatican City', 'Venezuela', 'Vietnam', 'Wallis and Futuna', 'Western Sahara', 'Yemen', 'Åland']


In [21]:
rename_map = {
    "Bahamas": "Bahamas, The",
    "Brunei": "Brunei Darussalam",
    "Cape Verde": "Cabo Verde",
    "Congo": "Congo, Rep.",
    "DR Congo": "Congo, Dem. Rep.",
    "Curaçao": "Curacao",
    "Czech Republic": "Czechia",
    "Egypt": "Egypt, Arab Rep.",
    "Federated States of Micronesia": "Micronesia, Fed. Sts.",
    "Gambia": "Gambia, The",
    "Hong Kong": "Hong Kong SAR, China",
    "Iran": "Iran, Islamic Rep.",
    "Ivory Coast": "Cote d'Ivoire",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Laos": "Lao PDR",
    "Macau": "Macao SAR, China",
    "Nauru": "Naoero",
    "North Korea": "Korea, Dem. People's Rep.",
    "Palestine": "West Bank and Gaza",
    "Puerto Rico": "Puerto Rico (US)",
    "Russia": "Russian Federation",
    "Saint Kitts and Nevis": "St. Kitts and Nevis",
    "Saint Lucia": "St. Lucia",
    "Saint Martin": "St. Martin (French part)",
    "Saint Vincent and the Grenadines": "St. Vincent and the Grenadines",
    "Sint Maarten": "Sint Maarten (Dutch part)",
    "Slovakia": "Slovak Republic",
    "Somalia": "Somalia, Fed. Rep.",
    "South Korea": "Korea, Rep.",
    "Syria": "Syrian Arab Republic",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "Turkey": "Turkiye",
    "U.S. Virgin Islands": "Virgin Islands (U.S.)",
    "Venezuela": "Venezuela, RB",
    "Vietnam": "Viet Nam",
    "Yemen": "Yemen, Rep.",
}

df["Country"] = df["Country"].replace(rename_map)
print(len(rename_map))

36


In [22]:
merged = pd.merge(df, wb, on="Country", how="outer", indicator=True)
print(merged["_merge"].value_counts())

_merge
both          216
left_only      28
right_only      1
Name: count, dtype: int64


In [23]:
final = pd.merge(df, wb, on="Country", how="inner")
print(final.shape)
print(final.columns.tolist())

(216, 11)
['Country', 'Density', 'Population', 'Area', 'iso3', 'year', 'gdp_per_capita', 'life_expectancy', 'unemployment', 'population', 'log_gdp']


In [24]:
final = final.rename(columns={
    "Population": "population_wiki",
    "population": "population_wb",
    "Density": "density_km2",
    "Area": "area_km2",
})
print(final.columns.tolist())

['Country', 'density_km2', 'population_wiki', 'area_km2', 'iso3', 'year', 'gdp_per_capita', 'life_expectancy', 'unemployment', 'population_wb', 'log_gdp']


In [25]:
print(final.isna().sum())
print(final.duplicated().sum())
print(final["Country"].duplicated().sum())

Country             0
density_km2         0
population_wiki     0
area_km2            0
iso3                0
year                0
gdp_per_capita     13
life_expectancy     0
unemployment       33
population_wb       0
log_gdp            13
dtype: int64
0
0


In [26]:
final.to_csv("final_dataset.csv", index=False)
print(final.shape)

(216, 11)
